[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/projects/10_lora-fine-tuning.ipynb)

*Industry Project 10 of 10 · AI / ML Engineer · Advanced · about 8–10 hours*

# Fine-Tune With LoRA: Adapt a Model to a New Domain

**The situation.** *ShopKart*'s ticket router sends each message to the billing, delivery, account or technical team. It was trained on English product reviews ("charged twice, refund please") and works well there. But customers now write in support chat like *"paisa debited upi still"* or *"otp nahi aaya, kyc blocked"*, and the router gets many of them wrong.

You have only ~160 labelled chat messages. Retraining from scratch isn't an option, so you'll **fine-tune** the existing model — and compare two ways:

- **Full fine-tuning**: update every weight.
- **LoRA**: freeze the weights and learn a small low-rank update, the technique used to adapt large language models cheaply.

The model here is small enough to train on a laptop in seconds, so you can implement LoRA's maths yourself and see exactly what libraries like PEFT do. The last section runs the same recipe on a real LLM with QLoRA.

**What you'll hand in:** a cleaned training set, a from-scratch LoRA trainer, a comparison table (accuracy on the new and old domains, trainable parameters) for the base model, LoRA and full fine-tuning, a merged adapter, a chosen rank, and a QLoRA script for a real LLM.

**Skills:** LoRA, full vs parameter-efficient fine-tuning, gradient descent, data cleaning and leakage, catastrophic forgetting, adapter merging, rank selection, QLoRA with PEFT/TRL

**How this works:** each milestone has a task, a code cell for you to finish, and a ✅ check. Hints and a reference solution are hidden until you open them. Try for at least 20 minutes before opening the solution — the struggle is where the learning happens.

## The data

| File | Rows | What |
|---|---|---|
| `ft_base_embed.csv` | one per word | the pre-trained **embedding matrix** E: a 32-number vector per word |
| `ft_base_head.csv` | 33 | the pre-trained **classification head** H (32 × 4) and, in the last row, the bias |
| `ft_old_train.csv` / `ft_old_test.csv` | 800 / 200 | the old domain (reviews) — used only to measure forgetting |
| `ft_new_train.csv` | ~180 | new-domain chat messages to fine-tune on (**messy**: blanks, duplicates, and some leaked test messages) |
| `ft_new_test.csv` | 200 | new-domain messages to evaluate on |

The model averages the embeddings of a message's words and feeds that through the head: logits = (x·E)·H + b, where x holds each word's share of the message. A large language model is the same idea with many more layers. LoRA adapts big matrices like E — exactly what you'll implement here.

In [ ]:
# ▶ Run this cell first: it loads the libraries, the data and the checker.
import math, os, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
RAW = "https://raw.githubusercontent.com/abpandey-dev/learn-python-from-scratch-to-advanced/main/projects/data/"


def load(name):
    """Read a project data file: from the local data/ folder if it is there, otherwise from GitHub."""
    local = Path("data") / name
    return pd.read_csv(local if local.exists() else RAW + name)


def _plain(x):
    if isinstance(x, pd.DataFrame):
        return {str(c): _plain(x[c].tolist()) for c in x.columns}
    if isinstance(x, pd.Series):
        return {_plain(k): _plain(v) for k, v in x.items()}
    if isinstance(x, np.ndarray):
        return _plain(x.tolist())
    if isinstance(x, np.generic):
        return _plain(x.item())
    if isinstance(x, (list, tuple)):
        return [_plain(v) for v in x]
    if isinstance(x, dict):
        return {_plain(k): _plain(v) for k, v in x.items()}
    return x


def _same(a, b, tol):
    if isinstance(b, bool) or isinstance(a, bool):
        return a == b
    if isinstance(b, (int, float)) and isinstance(a, (int, float)):
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol if tol > 1e-3 else 1e-6)
    if isinstance(b, list) and isinstance(a, list):
        return len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    if isinstance(b, dict) and isinstance(a, dict):
        return set(map(str, a)) == set(map(str, b)) and all(_same(a[k], b[k], tol) for k in a if k in b)
    return a == b


def _why(a, b):
    if a is None:
        return "your function returned None — finish the code and return the result."
    if type(a) != type(b) and not (isinstance(a, (int, float)) and isinstance(b, (int, float))):
        return f"expected a {type(b).__name__}, got a {type(a).__name__}."
    if isinstance(b, (list, dict)) and len(a) != len(b):
        return f"expected {len(b)} items, got {len(a)}."
    if isinstance(b, dict) and set(map(str, a)) != set(map(str, b)):
        return f"the keys should be {sorted(map(str, b))[:8]}."
    return "the values are not right yet. Re-read the task and the hints."


def check(name, fn, want, tol=1e-3):
    """Run your code for one milestone and compare it with the expected answer."""
    try:
        got = _plain(fn())
    except NotImplementedError:
        print(f"⬜ {name}: not started yet.")
        return
    except Exception as e:
        print(f"❌ {name}: your code raised {type(e).__name__}: {e}")
        return
    if _same(got, want, tol):
        print(f"✅ {name}: correct!")
        return
    print(f"❌ {name}: not yet — {_why(got, want)}")

import matplotlib.pyplot as plt
embed, head = load("ft_base_embed.csv"), load("ft_base_head.csv")
old_train, old_test = load("ft_old_train.csv"), load("ft_old_test.csv")
new_train_raw, new_test = load("ft_new_train.csv").fillna(""), load("ft_new_test.csv")
CLASSES = ["billing", "delivery", "account", "technical"]
VOCAB = embed.word.tolist()
E0 = embed.drop(columns="word").to_numpy()           # (vocabulary, 32) pre-trained embeddings
H0 = head[CLASSES].to_numpy()[:-1]                   # (32, 4) classification head
B0 = head[CLASSES].to_numpy()[-1]                    # (4,) bias
STEPS, LR, SCALE = 300, 1.0, 2.0                     # SCALE = alpha / r with alpha = 2r


def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def labels_of(df):
    return np.array([CLASSES.index(l) for l in df.label])


# Used by the checks: your functions chained together.
def data():
    tr = clean(new_train_raw, new_test)[0]
    return vectorise(tr.text), labels_of(tr), vectorise(new_test.text), labels_of(new_test), vectorise(old_test.text), labels_of(old_test)


def lora_result(r):
    X, y, Xt, yt, Xo, yo = data()
    A, B = train_lora(X, y, r)
    return [accuracy(E0 + SCALE * A @ B, H0, B0, Xt, yt), A.size + B.size]



def full_result():
    X, y, Xt, yt, Xo, yo = data()
    E, H, b = train_full(X, y)
    return [accuracy(E, H, b, Xt, yt), E.size + H.size + b.size]
print("Ready! Data loaded.")

## Milestone 1: Clean the fine-tuning data

Write `clean(train, test)` returning `(clean_train, {reason: count})`. Drop, checking in this order:
- `"empty"`: text that is blank after stripping,
- `"duplicate"`: an exact (text, label) repeat of an earlier row,
- `"leak"`: text that also appears in the **test** set — training on test messages would make your results look better than they are.

With only ~160 examples, every bad row matters.

In [ ]:
def clean(train, test):
    raise NotImplementedError

In [ ]:
check("Clean the fine-tuning data", lambda: [len(clean(new_train_raw, new_test)[0]), clean(new_train_raw, new_test)[1]], [160, {'leak': 8, 'duplicate': 12, 'empty': 3}], tol=1e-09)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def clean(train, test):
    test_texts = set(test.text)
    keep, reasons, seen = [], {}, set()
    for i, (text, label) in enumerate(zip(train.text, train.label)):
        if not str(text).strip():
            reason = "empty"
        elif (text, label) in seen:
            reason = "duplicate"
        elif text in test_texts:
            reason = "leak"
        else:
            reason = None
        seen.add((text, label))
        if reason:
            reasons[reason] = reasons.get(reason, 0) + 1
        else:
            keep.append(i)
    return train.iloc[keep].reset_index(drop=True), reasons
```

</details>

## Milestone 2: Turn text into features

Write `vectorise(texts)` returning a NumPy array of shape (number of texts, len(VOCAB)) where entry [i, j] is the **share** of text i's known words that are `VOCAB[j]` (count ÷ number of known words; split on spaces, ignore unknown words, leave all-zero rows as zeros). Multiplying this by E averages the word embeddings.

In [ ]:
def vectorise(texts):
    raise NotImplementedError

In [ ]:
check("Turn text into features", lambda: [list(vectorise(['upi debited debited', 'hello world']).shape), [round(v, 4) for v in sorted(set(vectorise(['upi debited debited']).ravel().tolist()))], vectorise(['hello world']).sum()], [[2, 68], [0.0, 0.3333, 0.6667], 0.0], tol=1e-09)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def vectorise(texts):
    index = {w: j for j, w in enumerate(VOCAB)}
    X = np.zeros((len(texts), len(VOCAB)))
    for i, t in enumerate(texts):
        for w in str(t).split():
            if w in index:
                X[i, index[w]] += 1
    totals = X.sum(axis=1, keepdims=True)
    return np.divide(X, totals, out=np.zeros_like(X), where=totals > 0)
```

</details>

## Milestone 3: How good is the base model?

Write `accuracy(E, H, b, X, y)` — the share of rows where `argmax((X @ E) @ H + b)` equals the label — rounded to 4.

The check measures the pre-trained model on the **old** test set and the **new** one. That gap is why you're fine-tuning.

In [ ]:
def accuracy(E, H, b, X, y):
    raise NotImplementedError

In [ ]:
check("How good is the base model?", lambda: (lambda X, y, Xt, yt, Xo, yo: [accuracy(E0, H0, B0, Xo, yo), accuracy(E0, H0, B0, Xt, yt)])(*data()), [0.98, 0.51], tol=1e-09)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def accuracy(E, H, b, X, y):
    return round(float(((X @ E @ H + b).argmax(axis=1) == y).mean()), 4)
```

</details>

## Milestone 4: Train a LoRA adapter yourself

Freeze `E0`, `H0` and `B0`. Learn A (vocabulary × r) and B (r × 32) so the embeddings become E0 + SCALE·A·B, and logits = X·(E0 + SCALE·A·B)·H0 + B0.

Write `train_lora(X, y, r, steps=STEPS, lr=LR)` returning `(A, B)`:
- initialise `A = np.random.default_rng(0).normal(0, 0.01, (len(VOCAB), r))` and `B = np.zeros((r, 32))`,
- each step: P = softmax(logits), G = (P − one_hot(y)) / n, U = G·H0ᵀ (the gradient reaching the embeddings), then
  `grad_B = SCALE · (X·A)ᵀ·U` and `grad_A = SCALE · Xᵀ·(U·Bᵀ)`, and subtract `lr ×` each (both from the old A and B).

The check trains with r = 4 and reports `[new-domain accuracy, trainable parameters]`.

<details>
<summary><b>💡 Hint</b></summary>

One-hot labels: `np.eye(4)[y]`. Compute `G` and `U` once per step and use them for both gradients.

</details>

In [ ]:
def train_lora(X, y, r, steps=STEPS, lr=LR):
    raise NotImplementedError

In [ ]:
check("Train a LoRA adapter yourself", lambda: lora_result(4), [0.865, 400], tol=0.02)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def train_lora(X, y, r, steps=STEPS, lr=LR):
    A = np.random.default_rng(0).normal(0, 0.01, (len(VOCAB), r))
    B = np.zeros((r, 32))
    Y = np.eye(4)[y]
    for _ in range(steps):
        P = softmax(X @ (E0 + SCALE * A @ B) @ H0 + B0)
        U = (P - Y) / len(X) @ H0.T
        grad_B = SCALE * (X @ A).T @ U
        grad_A = SCALE * X.T @ (U @ B.T)
        A, B = A - lr * grad_A, B - lr * grad_B
    return A, B
```

</details>

On the very first step A gets no gradient (B is zero), but B does — then A starts moving too. That's why LoRA can start at exactly the original model without being stuck.

## Milestone 5: Compare with full fine-tuning

Write `train_full(X, y, steps=STEPS, lr=LR)` that starts from copies of `E0`, `H0` and `B0` and updates **all** of them with gradient descent on the same loss: with h = X·E and G as before, `grad_H = hᵀ·G`, `grad_E = Xᵀ·(G·Hᵀ)`, `grad_b = Σ G over rows`. Return `(E, H, b)`.

The check returns `[new accuracy, trainable parameters]`. Compare with LoRA.

In [ ]:
def train_full(X, y, steps=STEPS, lr=LR):
    raise NotImplementedError

In [ ]:
check("Compare with full fine-tuning", lambda: full_result(), [0.905, 2308], tol=0.02)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def train_full(X, y, steps=STEPS, lr=LR):
    E, H, b, Y = E0.copy(), H0.copy(), B0.copy(), np.eye(4)[y]
    for _ in range(steps):
        h = X @ E
        G = (softmax(h @ H + b) - Y) / len(X)
        grad_H, grad_E = h.T @ G, X.T @ (G @ H.T)
        E, H, b = E - lr * grad_E, H - lr * grad_H, b - lr * G.sum(axis=0)
    return E, H, b
```

</details>

## Milestone 6: Did it forget the old domain? And merge the adapter

Write `forgetting_and_merge(r=4)` returning a dict:
- `old_base`, `old_lora`, `old_full`: accuracy on the **old** test set for the base model, the LoRA model and the fully fine-tuned model,
- `merge_identical`: whether predictions with the merged embeddings `E0 + SCALE·A·B` equal computing base and adapter separately (`X·E0 + SCALE·(X·A)·B`) on the new test set — the reason a merged LoRA costs nothing extra to serve.

In [ ]:
def forgetting_and_merge(r=4):
    raise NotImplementedError

In [ ]:
check("Did it forget the old domain? And merge the adapter", lambda: forgetting_and_merge(), {'old_base': 0.98, 'old_lora': 0.79, 'old_full': 0.98, 'merge_identical': True}, tol=0.02)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def forgetting_and_merge(r=4):
    X, y, Xt, yt, Xo, yo = data()
    A, B = train_lora(X, y, r)
    Ef, Hf, bf = train_full(X, y)
    merged = E0 + SCALE * A @ B
    separate = ((Xt @ E0 + SCALE * (Xt @ A) @ B) @ H0 + B0).argmax(axis=1)
    return {"old_base": accuracy(E0, H0, B0, Xo, yo), "old_lora": accuracy(merged, H0, B0, Xo, yo),
            "old_full": accuracy(Ef, Hf, bf, Xo, yo),
            "merge_identical": bool((separate == (Xt @ merged @ H0 + B0).argmax(axis=1)).all())}
```

</details>

Look closely: here LoRA forgot **more** of the old domain than full fine-tuning. The adapter could only change the word embeddings, and some words ("update", "box", "screen", "verify") mean different intents in the two domains — so it rewrote words the old domain relies on, while full fine-tuning could also adjust the head. "LoRA forgets less" is a tendency, not a law: always measure the old task after fine-tuning.

## Milestone 7: Choose the rank

Write `rank_sweep(ranks=(1, 2, 4, 8, 16), tol=0.01)` that trains LoRA at each rank and returns `(chosen rank, {rank: new-domain accuracy})` — the chosen rank is the **smallest** whose accuracy is within `tol` of the best. Smaller adapters are cheaper to store, load and serve.

In [ ]:
def rank_sweep(ranks=(1, 2, 4, 8, 16), tol=0.01):
    raise NotImplementedError

In [ ]:
check("Choose the rank", lambda: rank_sweep(), [4, {1: 0.72, 2: 0.845, 4: 0.865, 8: 0.87, 16: 0.87}], tol=0.02)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def rank_sweep(ranks=(1, 2, 4, 8, 16), tol=0.01):
    X, y, Xt, yt, Xo, yo = data()
    scores = {}
    for r in ranks:
        A, B = train_lora(X, y, r)
        scores[r] = accuracy(E0 + SCALE * A @ B, H0, B0, Xt, yt)
    best = max(scores.values())
    return [min(r for r in ranks if scores[r] >= best - tol - 1e-9), scores]
```

</details>

## A chart for your report

In [ ]:
try:
    X, y, Xt, yt, Xo, yo = data()
    A, B = train_lora(X, y, 4)
    Ef, Hf, bf = train_full(X, y)
    rows = {"base": (E0, H0, B0), "LoRA r=4": (E0 + SCALE * A @ B, H0, B0), "full fine-tune": (Ef, Hf, bf)}
    new = [accuracy(*m, Xt, yt) for m in rows.values()]
    old = [accuracy(*m, Xo, yo) for m in rows.values()]
    x = np.arange(3)
    plt.figure(figsize=(7, 3))
    plt.bar(x - 0.2, new, 0.4, label="new domain (chats)", color="#0f766e")
    plt.bar(x + 0.2, old, 0.4, label="old domain (reviews)", color="#94a3b8")
    plt.xticks(x, rows.keys()); plt.ylim(0, 1); plt.ylabel("accuracy"); plt.legend(); plt.title("Fine-tuning: gain on new data, cost on old")
    plt.show()
except NotImplementedError:
    print("Finish the milestones above first — this chart uses your functions.")

## Now do it on a real LLM with QLoRA

The same recipe — freeze the base, train low-rank adapters, evaluate on held-out data and the old task — applied to a 0.5B-parameter chat model, stored in 4 bits. Run this on a GPU runtime (in Colab: Runtime → Change runtime type → T4 GPU). It isn't run automatically here.

```python
# Real QLoRA fine-tuning of a small open LLM — run on a GPU (e.g. Colab T4/A100).
# pip install -U transformers peft trl bitsandbytes datasets accelerate
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

model_id = "Qwen/Qwen2.5-0.5B-Instruct"                       # small enough for a free GPU
tok = AutoTokenizer.from_pretrained(model_id)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map="auto")

rows = [{"messages": [{"role": "user", "content": f"Classify this support message: {t}"},
                      {"role": "assistant", "content": l}]} for t, l in zip(new_train.text, new_train.label)]
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM")
trainer = SFTTrainer(model=model, train_dataset=Dataset.from_list(rows), peft_config=lora, processing_class=tok,
                     args=SFTConfig(output_dir="out", num_train_epochs=3, learning_rate=2e-4,
                                    per_device_train_batch_size=8, logging_steps=10))
trainer.train()
trainer.save_model("intent-lora")        # saves only the small adapter
# Argument names change between library versions — check the TRL / PEFT docs for yours.
```

Then evaluate the tuned model on `new_test` by generating a label for each message and computing accuracy — and compare against simply prompting the base model with a few examples. If prompting gets close, you may not need fine-tuning at all.

## Present your work

Write a short fine-tuning report:

1. **Problem**: the domain shift, with the base model's old vs new accuracy.
2. **Data**: what you removed and why (especially the leaked test rows).
3. **Results table**: base, LoRA (chosen rank) and full fine-tuning — new-domain accuracy, old-domain accuracy, trainable parameters.
4. **Recommendation**: which you'd ship and why (accuracy, forgetting, storage, serving several customers' adapters on one base).
5. **Next steps**: more labelled chats, the QLoRA run on a real LLM, and a prompting baseline.

## Put it on your resume

Write bullets with **your own numbers** from the milestones above. A template:

- Implemented LoRA from scratch (forward pass, gradients, merging) and adapted a pre-trained intent classifier to a new domain, lifting accuracy from **X%** to **Y%** with **Z** trainable parameters
- Compared LoRA with full fine-tuning on accuracy, forgetting and parameter count, and chose the adapter rank with a sweep
- Prepared a fine-tuning dataset with leakage and duplicate checks, and wrote a QLoRA (PEFT + TRL) script for a 0.5B chat model

Only claim what you built and measured. Put the notebook on your GitHub with a short README: problem, data, approach, results, what you'd do next.

## Interview questions about this project

Practise answering these out loud in 1–2 minutes each:

1. Explain LoRA. Why does a low-rank update work, and why is B initialised to zero?
2. LoRA vs full fine-tuning: what did you see on accuracy, forgetting and parameter count?
3. What happens if test examples leak into the training data?
4. How do you choose the rank and the scaling factor alpha?
5. Why can a merged LoRA model be served at no extra cost? When would you keep adapters separate?
6. When would you choose prompting or RAG instead of fine-tuning?

## Take it further

- Add a DPO-style step: build preference pairs between correct and confusable labels and optimise the DPO loss.
- Try different learning rates and steps for LoRA vs full fine-tuning — which is more sensitive?
- Run the QLoRA script on a GPU and compare with a few-shot prompt of the base LLM.
- Serve two adapters (two 'customers') on one base model and switch between them per request.